In [161]:
import pandas as pd
import joblib
import requests
import numpy as np

# Importing and Defining API Fuction

In [162]:
# --- RENAMING  ---
rename_cols = {
    'web_name': 'Player Name',
    'position': 'Pos',
    'team': 'Team',
    'final_xp': 'Predicted Points',
    'value': 'Price (£m)',
    'rolling_3_total_points': 'Recent Form',
    'actual_points': 'Actual Points',
    'chance_of_playing': 'Availabilty %',
    'error': 'Diff',
    'starter_prob': 'Start %'
}

In [ ]:
def fetch_live_data():
    print("Fetching live FPL data...")
    url = "https://fantasy.premierleague.com/api/bootstrap-static/"
    r = requests.get(url)
    data = r.json()
    
    # Geting the Current Gameweek
    current_gw = next(e['id'] for e in data['events'] if e['is_current'])
    # Avoid division by zero for GW1
    current_gw = max(1, current_gw)

    # Processing Players
    df = pd.DataFrame(data['elements'])

    # Calculating Average Minutes
    df['minutes_per_game'] = (df['minutes'] / current_gw).fillna(0)

    # Using Average Minutes in last 3 games
    df['rolling_3_minutes'] = (df['minutes'] / current_gw).fillna(0)
    # Using Average Points in last 3 games
    df['rolling_3_total_points'] = pd.to_numeric(df['form'], errors='coerce').fillna(0)
    
    # Using Average ICT in last 3 games
    df['ict_index'] = pd.to_numeric(df['ict_index'], errors='coerce')
    df['rolling_3_ict_index'] = (df['ict_index'] / current_gw).fillna(0)
    
    # Creativity/Influence/Threat: Converting Total -> Average
    df['creativity'] = pd.to_numeric(df['creativity'], errors='coerce')
    df['influence'] = pd.to_numeric(df['influence'], errors='coerce')
    df['threat'] = pd.to_numeric(df['threat'], errors='coerce')
    
    df['rolling_3_creativity'] = (df['creativity'] / current_gw).fillna(0)
    df['rolling_3_influence'] = (df['influence'] / current_gw).fillna(0)
    df['rolling_3_threat'] = (df['threat'] / current_gw).fillna(0)
    df['chance_of_playing'] = df['chance_of_playing_next_round'].fillna(75)

    # Probability to start
    df['starter_prob'] = (df['minutes_per_game'] / 75).clip(0, 1) * 100
    
    # Value (Price)
    df['value'] = df['now_cost'] / 10
    
    # Helpers for display
    df['position'] = df['element_type'].map({1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'})
    
    return df, current_gw

In [164]:
# Fetching Points for a specific GW
def fetch_actual_points(gw):
    print(f"Fetching actual results for GW {gw}...")
    url = f"https://fantasy.premierleague.com/api/event/{gw}/live/"
    r = requests.get(url)
    if r.status_code != 200: 
        print("Could not fetch actual data (maybe future gameweek?).")
        return None
    
    data = r.json()
    actuals = []
    for player in data['elements']:
        actuals.append({
            'id': player['id'],
            'actual_points': player['stats']['total_points']
        })
    return pd.DataFrame(actuals)

In [165]:
# Loading Model
print("Loading Model...")
try:
    model = joblib.load("../Models/linear_reg_v1.pkl")
    print("✅ Model loaded successfully.")
except FileNotFoundError:
    print("❌ Error: Model file not found. Check path.")

Loading Model...
✅ Model loaded successfully.


In [166]:
# Fetch & Prepare Data
live_df, current_gw = fetch_live_data()
PREDICTION_GW = current_gw + 1

Fetching live FPL data...


In [170]:
# Predicting the data
print(f"Generating predictions for Gameweek {current_gw + 1}...")

# Features
features_for_model = [
    'rolling_3_minutes',
    'rolling_3_ict_index',
    'rolling_3_creativity',
    'rolling_3_influence',
    'rolling_3_threat',
    'rolling_3_total_points',
    'value'
]

# Creating Input Matrix
X = live_df[features_for_model].fillna(0)
live_df['predicted_xp'] = model.predict(X)

# Defcon Boost
def apply_boosts(row):
    pred = row['predicted_xp']
    
    # Defender Boost (Clean Sheet Potential)
    if row['position'] == 'DEF': pred = pred * 1.1
    # Goalkeeper Boost (Save Points)
    if row['position'] == 'GKP': pred = pred * 1.05

    # The "Availability" Check (Is he injured?)
    avail_factor = row['chance_of_playing'] / 100.0
    
    # The "Selection" Check (Is he a starter?)
    selection_factor = row['starter_prob'] / 100.0
    
    # Final Calculation:
    final_pred = pred * avail_factor * selection_factor
    
    return final_pred

live_df['final_xp'] = live_df.apply(apply_boosts, axis=1)
live_df.loc[live_df['chance_of_playing'] == 0, 'final_xp'] = 0
# Clamp to realistic FPL range
live_df['final_xp'] = live_df['final_xp'].clip(0, 15).round(2)
live_df['starter_prob'] = live_df['starter_prob'].round(0)

assert live_df['final_xp'].max() <= 15
assert live_df['final_xp'].min() >= 0

Generating predictions for Gameweek 23...


In [171]:
# Displaying Top Predictions
print(f"\n🏆 Top 15 Predicted Players for GW {current_gw + 1}:")
cols_to_show = ['web_name', 'position', 'team', 'final_xp', 'value', 'rolling_3_total_points']
top_picks = live_df.sort_values(by='final_xp', ascending=False).head(15)[cols_to_show]
display(top_picks.rename(columns=rename_cols))


🏆 Top 15 Predicted Players for GW 23:


,Player Name,Pos,Team,Predicted Points,Price (£m),Recent Form
202,Thiago,FWD,5,1.89,7.2,7.6
19,Rice,MID,1,1.81,7.4,6.5
187,Schade,MID,5,1.72,7.1,7.2
269,Enzo,MID,7,1.66,6.5,5.6
51,Rogers,MID,2,1.66,7.7,4.5
471,Semenyo,MID,13,1.61,7.6,4.6
502,Haaland,FWD,13,1.59,15.1,2.8
607,Gibbs-White,MID,16,1.53,7.3,4.3
169,Kelleher,GKP,5,1.51,4.6,5.2
450,Wirtz,MID,12,1.51,8.3,7.3


# ----------------------------------------------------
# 🛑 USER INPUT: Choose Gameweek
# ----------------------------------------------------

In [172]:
target_gw_input = input(f"Enter Gameweek to analyze): ")
if target_gw_input.strip() == "":
    target_gw = current_gw + 1
else:
    target_gw = int(target_gw_input)

Enter Gameweek to analyze):  7


In [173]:
# Mergeing with Actuals
actual_df = fetch_actual_points(target_gw)
cols_to_show = ['web_name', 'position', 'team', 'final_xp', 'value']

if actual_df is not None:
    # Merge prediction with reality
    live_df = pd.merge(live_df, actual_df, on='id', how='left')
    live_df['actual_points'] = live_df['actual_points'].fillna(0)
    
    # Calculating Error
    live_df['error'] = live_df['final_xp'] - live_df['actual_points']
    
    # Update columns to show
    cols_to_show += ['actual_points', 'error']

    print(f"\n✅ Comparing Model vs Reality for GW {target_gw}")
    # Displaying the comparison table sorted by Actual Points
    print(f"\n📊 Top Performers in GW {target_gw} (Actual vs Predicted):")
    comparison_table = live_df.sort_values(by='actual_points', ascending=False).head(15)[cols_to_show]
    display(comparison_table.rename(columns=rename_cols))
    
    # Calculate RMSE
    rmse = np.sqrt((live_df['error'] ** 2).mean())
    print(f"\n📉 RMSE for GW {target_gw}: {rmse:.2f}")

Fetching actual results for GW 7...

✅ Comparing Model vs Reality for GW 7

📊 Top Performers in GW 7 (Actual vs Predicted):


,Player Name,Pos,Team,Predicted Points,Price (£m),Actual Points,Diff
471,Semenyo,MID,13,1.61,7.6,18.0,-16.39
56,Malen,MID,2,0.00,5.1,15.0,-15.00
573,Bruno G.,MID,15,1.44,7.2,14.0,-12.56
692,Kudus,MID,18,0.00,6.4,12.0,-12.00
477,Gvardiol,DEF,13,0.00,6.0,12.0,-12.00
561,Burn,DEF,15,0.00,5.0,11.0,-11.00
218,Van Hecke,DEF,6,1.05,4.5,11.0,-9.95
19,Rice,MID,1,1.81,7.4,11.0,-9.19
7,J.Timber,DEF,1,1.48,6.3,11.0,-9.52
273,Caicedo,MID,7,0.64,5.7,10.0,-9.36



📉 RMSE for GW 7: 2.30


### ==========================================
# 🔎 INTERACTIVE PLAYER SEARCH
### ==========================================

In [174]:
prediction_df = live_df.copy()

# Remove backtest-only columns if they exist
prediction_df = prediction_df.drop(columns=['actual_points', 'error'], errors='ignore')

In [176]:
print("\n" + "="*40)
print(f"🔎 PLAYER SEARCH — GW {PREDICTION_GW}")
print("="*40)

player_name = input("Enter player name to search: ").strip()

print("="*40)
print("="*40)

print(f"Predicted Points of {player_name}")

if player_name:
    result = prediction_df[prediction_df['web_name'].str.contains(player_name, case=False, na=False)][
        ['web_name', 'team', 'position', 'final_xp', 'chance_of_playing', 'starter_prob', 'value', 'rolling_3_total_points']
    ]
    
    if not result.empty:
        print(f"\n✅ Prediction for GW {PREDICTION_GW}:")
        display(result.rename(columns=rename_cols))
    else:
        print(f"❌ Player '{player_name}' not found.")


🔎 PLAYER SEARCH — GW 23


Enter player name to search:  doku


Predicted Points of doku

✅ Prediction for GW 23:


,Player Name,Team,Pos,Predicted Points,Availabilty %,Start %,Price (£m),Recent Form
492,Doku,13,MID,0.47,100.0,70.0,6.4,2.4
